# OXIOW — Wan 2.2 Animate na T4 grátis (Colab)

**O que este notebook faz:** pega um **vídeo REAL** de uma pessoa falando e troca a
identidade dela pela **Margaret**, mantendo o movimento e a boca exatamente como no
original. É o caminho que a InfluAI usa (a frase do tutorial: *"identity never comes
from text, it comes from the reference images"*).

**Por que não é o S2V que eu usava antes:** o S2V **inventa** o movimento — e inventava
só "abre e fecha". O Animate **copia** o movimento de um vídeo real, então a articulação
vem de graça, de uma pessoa de verdade.

**Antes de rodar:** menu **Runtime → Alterar tipo de ambiente de execução → T4 GPU → Salvar**.
A célula 1 confere se veio T4 mesmo (o Colab às vezes entrega K80/P100).

**O que o notebook faz sozinho:** baixa os 3 arquivos que faltam do Hugging Face
(umt5 fp8 ~6,7 GB, dw-ll_ucoco, yolox_l), converte o VAE do Wan 2.1 para safetensors,
monta o ComfyUI e roda o workflow.

**Custo: R$ 0,00.** Nada de API paga, nada de MiniMax H3, nada de Seedance.


In [ ]:
# ── 1. AMBIENTE: confere a GPU (tem que ser T4 / sm_75) ──
import os, sys, subprocess, shutil, time, glob, json

def sh(cmd, timeout=5400, quiet=False, env=None):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       timeout=timeout, env=env)
    out = (p.stdout or '') + (p.stderr or '')
    if not quiet:
        print('\n'.join(out.strip().splitlines()[-14:]))
    return p.returncode, out

rc, _ = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader", quiet=True)
print("GPU:", _.strip() if _.strip() else "NENHUMA — ative a T4 em Runtime > Change runtime type")
_ok = 'T4' in _ or 'sm_75' in _ or '7.5' in _
print("\n>>> T4/sm_75?", "SIM" if _ok else "NAO — se for P100/K80, o torch nao roda. Troque o runtime.")

sh("df -h /content | tail -1")
sh("free -g | head -2")


In [ ]:
# ── 2. ARVORE DE TRABALHO + o que JA' temos (vem do Drive ou do dataset) ──
# No Colab o disco e' ~100 GB e a rede e' do Google: baixamos o que falta AQUI.
W = "/content/animate"
sh(f"mkdir -p {W}/models/{{diffusion_models,vae,text_encoders,sam2,loras,controlnet_aux,clip_vision}}")
sh(f"mkdir -p {W}/{{input,output,temp}}")
print("arvore criada em", W)

# o video REAL de referencia (a fonte do movimento). O dono vai subir 1 video aqui,
# ou usamos um dos que ja' capturamos do Instagram.
print('''
>>> PROXIMO PASSO MANUAL (1 vez): arraste um video REAL de pessoa falando para
    /content/animate/input/  —  ou rode a proxima celula e ela baixa um exemplo livre.
''')


In [ ]:
# ── 3. BAIXA O QUE FALTA DO HUGGING FACE (so' o que nao temos) ──
# Comfy-Org tem o repack do Wan 2.2 pronto para o ComfyUI (Apache-2.0, sem gate).
BASE = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"

FALTAM = [
  # (destino, url)
  (f"{W}/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
   f"{BASE}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
  (f"{W}/models/vae/wan_2.1_vae.safetensors",
   f"{BASE}/vae/wan_2.1_vae.safetensors"),
  (f"{W}/models/controlnet_aux/dw-ll_ucoco_384_bs5.torchscript.pt",
   "https://huggingface.co/hr16/DWPose-TorchScript-BatchSize5/resolve/main/dw-ll_ucoco_384_bs5.torchscript.pt"),
]

for dest, url in FALTAM:
    if os.path.exists(dest) and os.path.getsize(dest) > 1024*1024:
        print("ja' tem:", os.path.basename(dest)); continue
    print("baixando", os.path.basename(dest), "...")
    rc, out = sh(f"wget -q --show-progress -O '{dest}' '{url}'", timeout=1800)
    if os.path.exists(dest):
        print(f"  {os.path.getsize(dest)/1024/1024:.1f} MB")

# yolox (detector de pessoa) — o ComfyUI aceita .pt ou .onnx
sh(f"wget -q -O {W}/models/controlnet_aux/yolox_l.onnx "
   "https://huggingface.co/hr16/yolox-onnx/resolve/main/yolox_l.onnx", timeout=600)
print("\n>>> faltantes resolvidos. Agora o ComfyUI.")


In [ ]:
# ── 4. COMFYUI + os nos do Animate ──
os.chdir('/content')
if not os.path.isdir('/content/ComfyUI'):
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI", timeout=900)
sh("pip -q install -r /content/ComfyUI/requirements.txt", timeout=1800)
# nos de video do Wan (VHS) + GGUF (para o Q4) + aux (pose/deteccao)
DONOS = {'ComfyUI-VideoHelperSuite': 'Kosinkadink',
         'ComfyUI-GGUF': 'city96',
         'comfyui_controlnet_aux': 'Fannovel16'}
for repo, dono in DONOS.items():
    d = '/content/ComfyUI/custom_nodes/' + repo
    if not os.path.isdir(d):
        sh('git clone --depth 1 https://github.com/' + dono + '/' + repo + '.git ' + d, timeout=900)
        sh('pip -q install -r ' + d + '/requirements.txt', timeout=1200, quiet=True)
print("\n>>> ComfyUI pronto.")
print(">>> os modelos ficam em /content/animate/models — o ComfyUI le' por LINK, sem copiar:")
for sub in ['diffusion_models','vae','text_encoders','sam2','loras','controlnet_aux']:
    sh('mkdir -p /content/ComfyUI/models/' + sub, quiet=True)
    sh('ln -sfn ' + W + '/models/' + sub + ' /content/ComfyUI/models/' + sub + '/oxiow_' + sub, quiet=True)
sh("ls -la /content/ComfyUI/models/diffusion_models/ | tail -3")


In [ ]:
# ── 5. SOBE O Q4 (o nosso) + o SAM2 ──
# o Q4_K_M (10,7 GB) esta' no Oracle: o caminho e' pelo DRIVE (confiavel) ou re-baixar.
from google.colab import drive
drive.mount('/content/drive')
print('''
>>> MANUAL (1 vez): suba para o seu Drive, na pasta OXIOW/animate/:
      Wan2.2-Animate-14B-Q4_K_M.gguf   (10,7 GB)
      sam2_hiera_base_plus.safetensors (0,3 GB)
    (ou rode a celula abaixo para o drive procurar)
''')
for alvo in ["Wan2.2-Animate-14B-Q4_K_M.gguf", "sam2_hiera_base_plus.safetensors"]:
    achou = glob.glob(f"/content/drive/MyDrive/**/{alvo}", recursive=True)
    if achou:
        sub = "diffusion_models" if alvo.endswith(".gguf") else "sam2"
        sh(f"ln -sfn '{achou[0]}' {W}/models/{sub}/{alvo}")
        print("linkado:", alvo, "->", achou[0])
    else:
        print("NAO achei no Drive:", alvo)

sh(f"ls -la {W}/models/diffusion_models/ {W}/models/sam2/ {W}/models/text_encoders/ {W}/models/vae/")


In [ ]:
# ── 6. O TOKEN do ngrok (opcional) — para ver a interface do ComfyUI de fora ──
# SEM token o ComfyUI roda local no Colab (a interface aparece na celula 7).
# COM token voce acessa de qualquer lugar (o dono ja' usa ngrok). NUNCA cole o token
# aqui dentro se o notebook for compartilhado — use o cofre do Colab (chave ngrok).
try:
    from google.colab import userdata
    tok = userdata.get('NGROK_TOKEN')
except Exception:
    tok = None
if tok:
    sh("pip -q install pyngrok", timeout=600, quiet=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(tok)
    sh("pkill -f 'python.*main.py' 2>/dev/null || true")
    pub = ngrok.connect(8188).public_url
    print(">>> ComfyUI acessivel de fora em:", pub)
else:
    print(">>> sem NGROK_TOKEN no cofre do Colab — a interface abre local na celula 7")


In [ ]:
# ── 7. RODA O COMFYUI (deixa rodando; a interface abre no log) ──
# O ComfyUI serve em 127.0.0.1:8188. No Colab, o jeito de abrir e' pelo proxy local:
# rode esta celula em BACKGROUND (ela nao termina) e use a celula 8 para enviar o job.
import subprocess, time, sys
env = dict(os.environ); env['PYTHONUNBUFFERED'] = '1'
proc = subprocess.Popen(
    [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--dont-print-server'],
    cwd='/content/ComfyUI', stdout=open('/content/comfy.log','w'), stderr=subprocess.STDOUT, env=env)
for i in range(60):
    time.sleep(2)
    try:
        import urllib.request
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=4)
        print(">>> ComfyUI NO AR em http://127.0.0.1:8188 (pid", proc.pid, ")"); break
    except Exception:
        if i % 8 == 7: print('  ... subindo', (i+1)*2, 's')
else:
    print(">>> nao subiu; veja /content/comfy.log:")
    print(open('/content/comfy.log').read()[-1500:])


In [ ]:
# ── 8. O QUE ENVIAR: video de referencia (movimento) + as imagens da Margaret ──
# O workflow e' o "wan animate native.json" (62 nos): WanAnimateToVideo, Sam2Segmentation,
# FaceMaskFromPoseKeypoints. Ele troca a identidade do video de referencia pela Margaret.
import glob
print('MANUAL (1 vez): envie para /content/animate/ref/ as imagens da Margaret')
print('  (rosto + corpo + COSTAS — a receita dos 3 tutoriais)')
print('e para /content/animate/input/ o VIDEO REAL que da o movimento (5 a 15 s).')
print('')
for f in sorted(glob.glob('/content/animate/ref/*') + glob.glob('/content/animate/input/*')):
    mb = os.path.getsize(f)/1024/1024
    print('  %8.2f MB  %s' % (mb, f))
print('')
print('Quando os arquivos estiverem la, me diga que eu envio o workflow pela API do ComfyUI.')
print('Router do ComfyUI: http://127.0.0.1:8188')


## Como isto entra no fluxo da fábrica (o lugar único)

```
   ROTEIRO ........ Claude escreve (compliance: sem alegação médica implícita)
   VOZ ............ Qwen3-TTS (Apache-2.0) ou edge-tts
   VIDEO REAL ..... um vídeo de referência (o movimento) — 5 a 15 s
   ANIMATE ........ troca a identidade -> vídeo da Margaret com a boca CERTA
   MONTAGEM ....... ffmpeg: cortes, 9:16, legenda palavra-a-palavra, metadado limpo
   PUBLICAR ....... Zernio / o painel
```

**A diferença de fundo em relação ao que eu tentava antes:** nos motores anteriores eu
pedia ao modelo que **inventasse** a fala. Aqui ele **copia** o movimento de uma pessoa
de verdade — por isso a boca vem certa sem nenhum remendo depois.
